# Tutorial 10 — Learning and testing the pairwise couple

The classical state-space model is the special case of the Gaussian **pairwise Markov
model** with no *measurement back-action* (`A_xy = 0`) and no *observation memory*
(`A_yy = 0`). This notebook shows that these two couple-defining coefficients can be

1. **recovered** from data by a *partial* EM (the smoother is the E-step), from the
   classical initialisation `A_xy = A_yy = 0`;
2. **tested** — a likelihood-ratio test decides whether back-action is present at all,
   with the asymptotic law $\Lambda \to \chi^2_{pq}(\lambda)$, $\lambda = 2N\,\mathrm{KL_{rate}}$; and
3. **applied to real data**, where the same test becomes a *directional* (Granger) test,
   calibrated model-free by phase-randomised surrogates.

It mirrors, in miniature, Secs. IV–V of the paper *Smoothing, Learning, and Testing the
Gaussian Pairwise Markov Model* (Figs. 5–7). The full calibrated study is in
`experiments/em_identification.py`, `experiments/em_lrt.py` and `experiments/em_realdata.py`.

> Record lengths and seed counts here are kept small so the notebook runs in seconds;
> the scripts above use the full sizes. With `B` surrogates the smallest
attainable p-value is `1/(B+1)`.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
DATA = Path.cwd() / "data"          # small bundled inputs shipped with this notebook

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2, ncx2
from scipy.optimize import minimize_scalar
%matplotlib inline

from prg.classes.linear_pkf import Linear_PKF
from prg.classes.param_linear import ParamLinear
from prg.models.linear._amq import LinearAmQ
from prg.learning.em_partial_dynamics import estimate_dynamics_em, back_action_lrt

Q_TRUE = np.array([[0.10, 0.05], [0.05, 0.10]])   # correlated process noise (R_xy != 0)
AXX, AYX = 0.6, 0.3                                # blocks a classical model also has
N_DEMO = 300                                       # short records: the tutorial runs in seconds

def couple_param(A_xy, A_yy, Q=Q_TRUE):
    """A scalar linear pairwise model with the given back-action / obs-memory."""
    A = np.array([[AXX, A_xy], [AYX, A_yy]])
    m = LinearAmQ(1, 1, A=A, mQ=0.5 * (Q + Q.T) + 1e-9 * np.eye(2),
                  mz0=np.zeros((2, 1)), Pz0=np.eye(2), pairwiseModel=True)
    kw = m.get_params().copy(); kw.pop("dim_x"); kw.pop("dim_y")
    return ParamLinear(0, 1, 1, **kw)

CLASSICAL_INIT = couple_param(0.0, 0.0)            # A_xy = A_yy = 0

## 1. A couple with back-action

We simulate a scalar pairwise process whose latent state is partly driven by the
observed channel (`A_xy = 0.4`) and whose observation carries memory (`A_yy = 0.4`),
with correlated process noise. A classical model cannot express either effect.

In [ ]:
true = couple_param(0.4, 0.4)
data = Linear_PKF(true, sKey=0).simulate_N_data(N_DEMO)
print(f"simulated {len(data)} steps; the classical model would force A_xy = A_yy = 0")

## 2. Recovering the coupling by partial EM

`estimate_dynamics_em` holds `A_xx`, `A_yx`, `Q` fixed and learns `A_xy`, `A_yy` from
the classical initialisation. Each E-step is one variational (`VAR`) smoothing pass;
the M-step is a closed-form regression of the residual on the observed `y`. The
observed-data log-likelihood increases monotonically.

In [ ]:
res = estimate_dynamics_em(CLASSICAL_INIT, data, tol=1e-5, max_iter=200)
print(f"recovered  A_xy = {res.A_xy.item():.3f}   A_yy = {res.A_yy.item():.3f}   "
      f"(true 0.4 / 0.4;  converged in {res.n_iter} iters)")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(res.loglik, "o-", ms=3)
ax.set(xlabel="EM iteration", ylabel="observed-data log-likelihood",
       title="Partial EM climbs out of the classical model (monotone)")
fig.tight_layout(); plt.show()

## 3. Identifiability — `A_yy` recovers more tightly than `A_xy`

Neither coupling block is identifiable from the observed `y` alone; freezing `A_xx`,
`A_yx`, `Q` restores both, **provided the latent state is observable through the forward
path** `A_yx` (here `A_yx = 0.3 ≠ 0`, so it is). Even then the two are not equally well
determined: `A_yy` is a direct *observed-to-observed* coefficient, whereas `A_xy` acts
only through the latent channel — so across seeds `A_yy` concentrates more tightly.
(Paper Sec. IV-A; Fig. 5.)

In [ ]:
axy, ayy = [], []
for s in range(3):
    d = Linear_PKF(couple_param(0.4, 0.4), sKey=s).simulate_N_data(N_DEMO)
    r = estimate_dynamics_em(CLASSICAL_INIT, d, tol=1e-5, max_iter=200)
    axy.append(r.A_xy.item()); ayy.append(r.A_yy.item())
axy, ayy = np.array(axy), np.array(ayy)
print(f"A_xy = {axy.mean():.3f} +/- {axy.std():.3f}   (latent-mediated, weaker)")
print(f"A_yy = {ayy.mean():.3f} +/- {ayy.std():.3f}   (directly observed, tighter)")

fig, ax = plt.subplots(figsize=(6, 3))
ax.axhline(0.4, ls=":", color="k", lw=1, label="truth (0.4)")
ax.scatter(np.zeros_like(axy), axy, c="tab:red", label="A_xy (back-action)")
ax.scatter(np.ones_like(ayy), ayy, c="tab:green", label="A_yy (obs. memory)")
ax.set_xticks([0, 1]); ax.set_xticklabels(["A_xy", "A_yy"]); ax.set_xlim(-0.5, 1.5)
ax.set(ylabel=f"estimate over 3 seeds (N={N_DEMO})",
       title="A_yy is recovered more tightly than A_xy")
ax.legend(fontsize=8); fig.tight_layout(); plt.show()

## 4. Testing for back-action (likelihood-ratio test)

Is back-action present at all? Test `H0: A_xy = 0` against `A_xy != 0`. Both models
are fit by EM (`A_yy` a free nuisance in each); the statistic
`Lambda = 2[ell(free) - ell(A_xy=0)]` is asymptotically chi-square with
`dim_x * dim_y = 1` degree of freedom under `H0`.

In [ ]:
data_ba = Linear_PKF(couple_param(0.4, 0.4), sKey=0).simulate_N_data(N_DEMO)  # back-action present
data_no = Linear_PKF(couple_param(0.0, 0.4), sKey=0).simulate_N_data(N_DEMO)  # back-action absent

lrt_ba = back_action_lrt(CLASSICAL_INIT, data_ba, tol=1e-5)
lrt_no = back_action_lrt(CLASSICAL_INIT, data_no, tol=1e-5)
print(f"back-action present : Lambda = {lrt_ba.stat:8.2f}   p = {lrt_ba.pvalue:.1e}   -> reject H0")
print(f"back-action absent  : Lambda = {lrt_no.stat:8.2f}   p = {lrt_no.pvalue:.3f}     -> keep H0")
print(f"(chi-square_{lrt_ba.dof} critical value at 5% = {chi2.ppf(0.95, lrt_ba.dof):.2f})")

## 5. The law of the statistic: $\Lambda \to \chi^2_{pq}(\lambda)$

Under `H0` the statistic follows $\chi^2_{pq}$; under an alternative it is *noncentral*
$\chi^2_{pq}(\lambda)$ with $\lambda = 2N\,\mathrm{KL_{rate}}$, the spectral
Kullback–Leibler rate between the couple's `y`-spectrum and its best `A_xy = 0` fit
(paper Prop. 1). That noncentrality is a **spectral integral** — no EM — so the whole
power curve is analytic. We overlay it on a cached Monte-Carlo `H0` null (350 draws,
produced by `experiments/em_lrt.py`; recomputing it here would need hundreds of EM fits).

In [ ]:
# spectral noncentrality lambda(A_xy) = 2N * min_{A_yy} KL_rate  (paper Prop. 1) -- no EM
WGRID = np.linspace(-np.pi, np.pi, 4096, endpoint=False)
def y_spectrum(A):
    z = np.exp(-1j * WGRID); axx, axy, ayx, ayy = A.ravel()
    det = (1 - axx * z) * (1 - ayy * z) - axy * ayx * z * z
    a = ayx * z; b = 1 - axx * z
    num = (Q_TRUE[0, 0] * np.abs(a)**2 + Q_TRUE[1, 1] * np.abs(b)**2
           + 2 * Q_TRUE[0, 1] * np.real(a * np.conj(b)))
    return num / np.abs(det)**2
def kl_rate(f, g):
    r = f / g; return float(np.mean(r - 1.0 - np.log(r)) * 0.5)
def predicted_lambda(axy, ayy_true, N):
    f = y_spectrum(np.array([[AXX, axy], [AYX, ayy_true]]))
    r = minimize_scalar(lambda ayy: kl_rate(f, y_spectrum(np.array([[AXX, 0.0], [AYX, ayy]]))),
                        bounds=(-0.95, 0.95), method="bounded")
    return 2 * N * r.fun

cache = np.load(DATA / "em_lrt_null.npz")            # H0 draws precomputed by em_lrt.py
lam0, Ncache, crit = cache["lam0"], int(cache["N"]), float(cache["crit"])
print(f"cached H0: {lam0.size} draws at N={Ncache}; mean {lam0.mean():.3f} (chi2_1 mean = 1), "
      f"empirical size {np.mean(lam0 > crit):.3f} at alpha=0.05")

axy_grid = np.linspace(0.0, 0.45, 60)
pow_pred = ncx2.sf(crit, 1, [predicted_lambda(a, 0.4, Ncache) for a in axy_grid])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.2))
xx = np.linspace(0.02, 12, 300)
a1.hist(lam0, bins=40, range=(0, 12), density=True, color="tab:red", alpha=0.55, label="MC null")
a1.plot(xx, chi2.pdf(xx, 1), "tab:blue", lw=1.8, label=r"$\chi^2_1$")
a1.axvline(crit, ls=":", color="0.4", label=f"reject if $\\Lambda>{crit:.2f}$")
a1.set(xlabel=r"$\Lambda$", ylabel="density", ylim=(0, 1.0), title="(a) null distribution")
a1.legend(fontsize=7)
a2.plot(axy_grid, pow_pred, "tab:blue", lw=1.8, label=r"predicted $\chi^2_1(\lambda)$")
a2.axhline(0.05, ls=":", color="0.4", label=r"$\alpha=0.05$")
a2.set(xlabel=r"back-action $A_{xy}$", ylabel="power", ylim=(-0.03, 1.03),
       title=r"(b) power, $\lambda=2N\,\mathrm{KL_{rate}}$")
a2.legend(fontsize=7, loc="lower right")
fig.tight_layout(); plt.show()

## 6. On real data: a directional test with surrogate calibration

With **both** series observed there is no latent gauge, and testing `A_xy = 0` in the
bivariate VAR(1) is a *Granger* test: does the past of `Y` help predict `X`? On the
algae–rotifer chemostat (a known predator–prey couple) we test both directions. The
$\chi^2$ approximation over-rejects on short real series, so we calibrate the null
**model-free** by phase-randomised surrogates of the driver — spectrum preserved,
cross-coupling destroyed (paper Sec. IV-D). With `B` surrogates the smallest
attainable p-value is `1/(B+1)`.


In [ ]:
import pandas as pd
d = pd.read_csv(DATA / "chemostat_C1.csv")
def zlog(a):
    a = np.log(np.asarray(a, float)); return (a - a.mean()) / a.std(ddof=1)
algae, rot = zlog(d["X0"]), zlog(d["Y0"])

def _ll(Z0, Z1, A, S):
    E = Z1 - Z0 @ A.T; Si = np.linalg.inv(S)
    _, ld = np.linalg.slogdet(S)
    return -0.5 * (len(Z0) * (2 * np.log(2 * np.pi) + ld) + np.sum((E @ Si) * E))
def fit_full(Z):
    Z0, Z1 = Z[:-1], Z[1:]
    A = (Z1.T @ Z0) @ np.linalg.inv(Z0.T @ Z0); S = ((Z1 - Z0 @ A.T).T @ (Z1 - Z0 @ A.T)) / len(Z0)
    return _ll(Z0, Z1, A, S)
def fit_restr(Z, iters=100):                          # ML with A[0,1]=0 (Y absent from X-eqn)
    Z0, Z1 = Z[:-1], Z[1:]; X0, Y0, X1, Y1 = Z0[:, 0], Z0[:, 1], Z1[:, 0], Z1[:, 1]; T = len(Z0)
    axx = (X0 @ X1) / (X0 @ X0); G = np.column_stack([X0, Y0]); ayx, ayy = np.linalg.solve(G.T @ G, G.T @ Y1)
    for _ in range(iters):
        ex, ey = X1 - axx * X0, Y1 - ayx * X0 - ayy * Y0
        S = np.array([[ex @ ex, ex @ ey], [ex @ ey, ey @ ey]]) / T
        w = np.linalg.inv(S); w00, w01, w11 = w[0, 0], w[0, 1], w[1, 1]
        sxx, sxy, syy = X0 @ X0, X0 @ Y0, Y0 @ Y0
        M = np.array([[sxx * w00, sxx * w01, sxy * w01], [sxx * w01, sxx * w11, sxy * w11],
                      [sxy * w01, sxy * w11, syy * w11]])
        v = np.array([X0 @ (w00 * X1 + w01 * Y1), X0 @ (w01 * X1 + w11 * Y1), Y0 @ (w01 * X1 + w11 * Y1)])
        axx, ayx, ayy = np.linalg.solve(M, v)
    return _ll(Z0, Z1, np.array([[axx, 0.0], [ayx, ayy]]), S)
def lam(Z): return max(2.0 * (fit_full(Z) - fit_restr(Z)), 0.0)
def phase_randomize(y, rng):
    m = y.mean(); Y = np.fft.rfft(y - m); ph = np.exp(1j * rng.uniform(0, 2 * np.pi, Y.shape[0]))
    ph[0] = 1.0
    if len(y) % 2 == 0: ph[-1] = 1.0
    return np.fft.irfft(Y * ph, n=len(y)) + m
def surrogate_test(X, Y, B=199, seed=1):              # H0: Y does not drive X
    obs = lam(np.column_stack([X, Y])); rng = np.random.default_rng(seed)
    null = np.array([lam(np.column_stack([X, phase_randomize(Y, rng)])) for _ in range(B)])
    return obs, (1 + int(np.sum(null >= obs))) / (1 + B)

for lab, X, Y in [("rotifers -> algae", algae, rot), ("algae -> rotifers", rot, algae)]:
    L, p = surrogate_test(X, Y)
    print(f"{lab:20s}  Lambda = {L:6.1f}   p_surrogate = {p:.3f}   "
          f"-> {'reject' if p < 0.05 else 'keep'} H0")

## Going further

- **`experiments/em_identification.py`**, **`experiments/em_lrt.py`** — the full
  calibrated study behind Figs. 5–6 (50 seeds for recovery; 350 seeds for the
  $\chi^2_1$ null and the power curve; empirical size 0.037 at $\alpha=0.05$).
- **`experiments/em_realdata.py`** — the full real-data study (Fig. 7, Table): chemostat,
  S&P 500 and wind, both directions, surrogate + held-out predictive tests.
- **`prg.learning.em_partial_dynamics`** — `estimate_dynamics_em` and `back_action_lrt`
  used here; **`prg.learning.em_partial_noise`** learns the noise block `Q` instead.
- Tutorials **07** and **09** cover the six linear smoothers that supply the E-step.